# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Houssem-Bjaoui/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Plain-language data contract (Lane 2: Refresh / Content Opportunity Scoring)**

- **One row means:** one `client_hash_id` × `content_hash_id` page snapshot on one `report_date` from the daily fact table.
- **Table(s) used:** `fact_content_daily_performance` (primary signals + proxy trend fields), joined with `dim_content` (content metadata like `word_count`).
- **Time window used:** development month **March 2026** (`2026-03-01` to `2026-03-31`), with a decision snapshot on `2026-03-31` for quick scoring.
- **What is predicted/ranked:** a decision-support ranking of pages likely to need refresh review, using proxy label `trend_direction = 'down'` at the March snapshot.
- **One deliberate exclusion:** June 2026 (`fact_content_daily_performance_sample`) is excluded from label logic because it is the sealed final month; using it in development would leak evaluation context.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not HF_TOKEN:
    print('HF_TOKEN not found. Add it as an environment variable or Colab Secret, then Run All again.')
    HAS_WAREHOUSE = False
else:
    HAS_WAREHOUSE = True

if HAS_WAREHOUSE:
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    REL = 'hf://datasets/FlyRank/internship-warehouse'
    TABLES = {
        'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
        'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    }
    DEV_START = '2026-03-01'
    DEV_END = '2026-03-31'
    DECISION_DATE = '2026-03-31'
    print('Connected. Development month:', DEV_START, 'to', DEV_END)


HF_TOKEN not found. Add it as an environment variable or Colab Secret, then Run All again.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields (max 5, honest)
1. `impressions_prev_30d` (feature: demand baseline)  
   **Knowable at the decision moment because** it is measured in the prior 30-day window already completed by `report_date`.
2. `clicks_prev_30d` (feature: click demand baseline)  
   **Knowable at the decision moment because** it is observed before or at the same snapshot date.
3. `gsc_avg_position` (feature: search visibility context)  
   **Knowable at the decision moment because** it is an observed ranking signal already logged in the warehouse by the snapshot date.
4. `days_since_last_update` (feature: freshness recency)  
   **Knowable at the decision moment because** content update recency is known immediately on that day.
5. `word_count` (feature: content depth proxy)  
   **Knowable at the decision moment because** page length metadata exists at decision time and does not require future outcomes.

### Label / proxy field
- `is_down_proxy = 1(trend_direction = 'down')` at `report_date='2026-03-31'`.

### Context fields (not model inputs)
- `client_hash_id`, `content_hash_id`, `report_date`.

### Excluded fields
- `trend_direction` and `trend_pct` are excluded as honest features because the label is derived from the same trend logic.
- `fact_content_daily_performance_sample` (June 2026) is excluded from development label logic because it is the sealed final month.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k=200):
    order = np.argsort(-scores)
    top = order[: min(k, len(order))]
    return float(np.mean(y_true[top])) if len(top) else np.nan

if HAS_WAREHOUSE:
    snapshot = con.sql(f"""
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            f.impressions_prev_30d,
            f.clicks_prev_30d,
            f.gsc_avg_position,
            f.days_since_last_update,
            f.trend_direction,
            d.word_count
        FROM {TABLES['fact_daily']} f
        LEFT JOIN {TABLES['dim_content']} d
          ON f.content_hash_id = d.content_hash_id
        WHERE f.report_date = DATE '{DECISION_DATE}'
          AND f.impressions_prev_30d IS NOT NULL
    """).df()

    snapshot['is_down_proxy'] = (snapshot['trend_direction'] == 'down').astype(int)
    snapshot['ctr_prev_30d'] = np.where(
        snapshot['impressions_prev_30d'] > 0,
        snapshot['clicks_prev_30d'] / snapshot['impressions_prev_30d'],
        0.0,
    )

    feature_cols_honest = [
        'impressions_prev_30d',
        'ctr_prev_30d',
        'gsc_avg_position',
        'days_since_last_update',
        'word_count',
    ]

    model_df = snapshot[feature_cols_honest + ['is_down_proxy']].copy()
    model_df = model_df.replace([np.inf, -np.inf], np.nan).fillna(0)

    X = model_df[feature_cols_honest].values
    y = model_df['is_down_proxy'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    honest_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    honest_model.fit(X_train, y_train)
    honest_scores = honest_model.predict_proba(X_test)[:, 1]

    honest_auc = roc_auc_score(y_test, honest_scores)
    honest_p_at_200 = precision_at_k(y_test, honest_scores, k=200)

    leaked_feature = snapshot['is_down_proxy'].values.reshape(-1, 1)
    X_leak = np.hstack([X, leaked_feature])
    Xl_train, Xl_test, yl_train, yl_test = train_test_split(
        X_leak, y, test_size=0.3, random_state=42, stratify=y
    )

    leaked_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    leaked_model.fit(Xl_train, yl_train)
    leaked_scores = leaked_model.predict_proba(Xl_test)[:, 1]

    leaked_auc = roc_auc_score(yl_test, leaked_scores)
    leaked_p_at_200 = precision_at_k(yl_test, leaked_scores, k=200)

    print('Rows at decision snapshot:', len(snapshot))
    print('Positive proxy rate (down):', round(snapshot['is_down_proxy'].mean() * 100, 2), '%')
    print('\nHonest model (no leaked feature):')
    print('  ROC-AUC:', round(honest_auc, 4))
    print('  Precision@200:', round(honest_p_at_200, 4))
    print('\nLeaked model (+ label-derived feature):')
    print('  ROC-AUC:', round(leaked_auc, 4))
    print('  Precision@200:', round(leaked_p_at_200, 4))
    print('\nLeakage explanation: the leaked feature is the label itself, so the score becomes artificially near-perfect.')
    print('Final result kept for Lane 2: the honest model metrics above (without leaked feature).')


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Exactly three verification queries are used below:
1. Grain check
2. Row count + date span check
3. Availability check using `IS TRUE`


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
if HAS_WAREHOUSE:
    q1 = f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS dup_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '{DEV_START}' AND DATE '{DEV_END}'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
    """

    q2 = f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '{DEV_START}' AND DATE '{DEV_END}'
    """

    q3 = f"""
    SELECT
        COUNT(*) AS rows_total,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / NULLIF(COUNT(*), 0),
            2
        ) AS pct_ga4_available
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '{DEV_START}' AND DATE '{DEV_END}'
    """

    print('Query 1 — verify grain (expect empty result):')
    display(con.sql(q1).df())

    print('Query 2 — verify row count and date span:')
    display(con.sql(q2).df())

    print('Query 3 — verify availability using IS TRUE:')
    display(con.sql(q3).df())


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One important limitation of this selected slice: **March 2026 is one development month from an unbalanced panel**, so behavior can differ by client history depth and measurement coverage. This means scores are useful for decision-support ranking, but they are not causal proof that refreshing content will cause recovery.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
if HAS_WAREHOUSE:
    limit_check = con.sql(f"""
        SELECT
            COUNT(*) AS clients_total,
            MIN(gsc_data_start) AS min_gsc_start,
            MAX(gsc_data_start) AS max_gsc_start,
            MIN(ga4_data_start) AS min_ga4_start,
            MAX(ga4_data_start) AS max_ga4_start
        FROM read_parquet('{REL}/dim_clients.parquet')
    """).df()
    display(limit_check)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
